# 🧭 Bússola Pública — Etapa 1: Exploração da API da Câmara dos Deputados

**Objetivo:** entender o terreno antes de escrever código de produção.  
**Não é para produção.** É para responder perguntas:
- O que cada endpoint retorna?
- Como funciona a paginação?
- Quais campos vêm nulos?
- Quais campos são chaves de relacionamento entre tabelas?

**Documentação oficial:** https://dadosabertos.camara.leg.br/swagger/api.html  
**Base URL:** `https://dadosabertos.camara.leg.br/api/v2`


# 0. Setup

In [2]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta

pd.set_option('display.max_columns', None)   # mostra todas as colunas
pd.set_option('display.max_colwidth', 80)    # não trunca texto nas células

BASE_URL = "https://dadosabertos.camara.leg.br/api/v2"
HEADERS  = {"Accept": "application/json"}

# Recorte temporal: últimos 30 dias
DATA_FIM    = datetime.today().strftime("%Y-%m-%d")
DATA_INICIO = (datetime.today() - timedelta(days=30)).strftime("%Y-%m-%d")

print(f"Período de análise: {DATA_INICIO} → {DATA_FIM}")

Período de análise: 2026-04-24 → 2026-05-24


## 1. Utilitário de chamada

Antes de explorar cada endpoint, criamos uma função auxiliar simples.  
Ela **não** é o código de produção — é só para não repetir boilerplate no notebook.

In [3]:
def get_endpoint(path: str, params: dict = {}) -> dict:
    """
    Faz uma chamada GET ao endpoint e retorna o JSON completo.
    Imprime status e quantidade de itens retornados para diagnóstico rápido.
    """
    url = f"{BASE_URL}{path}"
    resp = requests.get(url, headers=HEADERS, params=params, timeout=15)
    
    print(f"[{resp.status_code}] {resp.url}")
    
    if resp.status_code != 200:
        print(f"  ⚠️  Erro: {resp.text[:200]}")
        return {}
    
    data = resp.json()
    n_items = len(data.get("dados", []))
    print(f"  → {n_items} itens retornados nessa página")
    return data


def normalizar(data: dict) -> pd.DataFrame:
    """
    Converte o campo 'dados' da resposta em um DataFrame normalizado.
    """
    return pd.json_normalize(data.get("dados", []))

---
## 2. Explorando `/partidos`

Começo pelos partidos porque:
- Volume pequeno (poucos registros)
- Não precisa de filtro de data
- É uma tabela dimensão que vai se relacionar com deputados

In [4]:
# Primeira chamada — página 1, poucos itens para ver a estrutura

raw_partidos = get_endpoint("/partidos", params={"pagina": 1,
                                                 'itens': 100, 
                                                 "ordem": "ASC", 
                                                 "ordenarPor": "sigla"
                                                 })
# Inspeciona a estrutura bruta da resposta
print("\nChaves do JSON raíz:", list(raw_partidos.keys()))

[200] https://dadosabertos.camara.leg.br/api/v2/partidos?pagina=1&itens=100&ordem=ASC&ordenarPor=sigla
  → 21 itens retornados nessa página

Chaves do JSON raíz: ['dados', 'links']


In [5]:
#Inspeciona os 'links' - a API usa HATEOAS para paginação
print("Links de paginação:")
for link in raw_partidos.get("links", []):
    print(f" {link}")

Links de paginação:
 {'rel': 'self', 'href': 'https://dadosabertos.camara.leg.br/api/v2/partidos?pagina=1&itens=100&ordem=ASC&ordenarPor=sigla'}
 {'rel': 'first', 'href': 'https://dadosabertos.camara.leg.br/api/v2/partidos?ordem=ASC&ordenarPor=sigla&pagina=1&itens=100'}
 {'rel': 'last', 'href': 'https://dadosabertos.camara.leg.br/api/v2/partidos?ordem=ASC&ordenarPor=sigla&pagina=1&itens=100'}


In [6]:
df_partidos = normalizar(raw_partidos)
print(f'Shape: {df_partidos.shape}')
df_partidos.head(10)

Shape: (21, 4)


,id,sigla,nome,uri
0,36898,AVANTE,Avante,https://dadosabertos.camara.leg.br/api/v2/partidos/36898
1,37905,CIDADANIA,Cidadania,https://dadosabertos.camara.leg.br/api/v2/partidos/37905
2,36899,MDB,Movimento Democrático Brasileiro,https://dadosabertos.camara.leg.br/api/v2/partidos/36899
3,38011,MISSÃO,Partido Missão,https://dadosabertos.camara.leg.br/api/v2/partidos/38011
4,37901,NOVO,Partido Novo,https://dadosabertos.camara.leg.br/api/v2/partidos/37901
5,36779,PCdoB,Partido Comunista do Brasil,https://dadosabertos.camara.leg.br/api/v2/partidos/36779
6,36786,PDT,Partido Democrático Trabalhista,https://dadosabertos.camara.leg.br/api/v2/partidos/36786
7,37906,PL,Partido Liberal,https://dadosabertos.camara.leg.br/api/v2/partidos/37906
8,36896,PODE,Podemos,https://dadosabertos.camara.leg.br/api/v2/partidos/36896
9,37903,PP,Progressistas,https://dadosabertos.camara.leg.br/api/v2/partidos/37903


In [7]:
#Análise de nulos - Campos que vem vazios com frequência
print('Campos e % de nulos:')
print(df_partidos.isnull().mean().sort_values(ascending=False).to_string())

Campos e % de nulos:
id       0.0
sigla    0.0
nome     0.0
uri      0.0


In [8]:
# Olha um registro completo para entender todos os campos disponíveis
print("Exemplo de um partido (raw JSON):")
print(json.dumps(raw_partidos["dados"][0], indent=2, ensure_ascii=False))

Exemplo de um partido (raw JSON):
{
  "id": 36898,
  "sigla": "AVANTE",
  "nome": "Avante",
  "uri": "https://dadosabertos.camara.leg.br/api/v2/partidos/36898"
}


### 📝 Observações sobre `/partidos`

> - Campos disponíveis: `id`, `sigla`, `nome`, `uri`
> - Campos nulos: `0`
> - Chave de relacionamento com deputados: `sigla` ou `id`
> - Volume total (precisa de paginação?): `21 itens totais, não precisa de páginação se usarmos o rate maximo 100`
> - Decisão: quais campos vou persistir no banco? `id`, `sigla`, `nome`

---
## 3. Explorando `/deputados`

Os 513 deputados em exercício. 

In [9]:
# Primeira página de deputados
raw_deputados = get_endpoint("/deputados", params={"pagina": 1, "itens": 100, "ordem": "ASC", "ordenarPor": "nome"})

df_deputados = normalizar(raw_deputados)
print(f"Shape (1 página): {df_deputados.shape}")
df_deputados.head()

[200] https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=1&itens=100&ordem=ASC&ordenarPor=nome
  → 100 itens retornados nessa página
Shape (1 página): (100, 9)


,id,uri,nome,siglaPartido,uriPartido,siglaUf,idLegislatura,urlFoto,email
0,204379,https://dadosabertos.camara.leg.br/api/v2/deputados/204379,Acácio Favacho,MDB,https://dadosabertos.camara.leg.br/api/v2/partidos/36899,AP,57,https://www.camara.leg.br/internet/deputado/bandep/204379.jpg,dep.acaciofavacho@camara.leg.br
1,220714,https://dadosabertos.camara.leg.br/api/v2/deputados/220714,Adail Filho,MDB,https://dadosabertos.camara.leg.br/api/v2/partidos/36899,AM,57,https://www.camara.leg.br/internet/deputado/bandep/220714.jpg,dep.adailfilho@camara.leg.br
2,221328,https://dadosabertos.camara.leg.br/api/v2/deputados/221328,Adilson Barroso,PL,https://dadosabertos.camara.leg.br/api/v2/partidos/37906,SP,57,https://www.camara.leg.br/internet/deputado/bandep/221328.jpg,dep.adilsonbarroso@camara.leg.br
3,204560,https://dadosabertos.camara.leg.br/api/v2/deputados/204560,Adolfo Viana,PSDB,https://dadosabertos.camara.leg.br/api/v2/partidos/36835,BA,57,https://www.camara.leg.br/internet/deputado/bandep/204560.jpg,dep.adolfoviana@camara.leg.br
4,204528,https://dadosabertos.camara.leg.br/api/v2/deputados/204528,Adriana Ventura,NOVO,https://dadosabertos.camara.leg.br/api/v2/partidos/37901,SP,57,https://www.camara.leg.br/internet/deputado/bandep/204528.jpg,dep.adrianaventura@camara.leg.br


In [10]:
# Inspeciona os tipos de dado de cada coluna
print("Tipos de dado:")
print(df_deputados.dtypes.to_string())

Tipos de dado:
id               int64
uri                str
nome               str
siglaPartido       str
uriPartido         str
siglaUf            str
idLegislatura    int64
urlFoto            str
email              str


In [11]:
# PAGINAÇÃO — entendendo quantas páginas existem
# Pega uma página grande para estimar o total
teste_pag = get_endpoint("/deputados", params={"pagina": 1, "itens": 100})
n_pagina_1 = len(teste_pag.get("dados", []))
print(f"Itens na página 1 (max=100): {n_pagina_1}")

teste_pag2 = get_endpoint("/deputados", params={"pagina": 2, "itens": 100})
n_pagina_2 = len(teste_pag2.get("dados", []))
print(f"Itens na página 2 (max=100): {n_pagina_2}")

teste_pag2 = get_endpoint("/deputados", params={"pagina": 6, "itens": 100})
n_pagina_2 = len(teste_pag2.get("dados", []))
print(f"Itens na página 2 (max=100): {n_pagina_2}")

print(f"\nTotal estimado: ~{n_pagina_1 + n_pagina_2} deputados")
print("Conclusão: precisa de paginação?",(n_pagina_2 > 0))

[200] https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=1&itens=100
  → 100 itens retornados nessa página
Itens na página 1 (max=100): 100
[200] https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=2&itens=100
  → 100 itens retornados nessa página
Itens na página 2 (max=100): 100
[200] https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=6&itens=100
  → 13 itens retornados nessa página
Itens na página 2 (max=100): 13

Total estimado: ~113 deputados
Conclusão: precisa de paginação? True


In [12]:
# Análise de nulos
print("Campos e % de nulos:")
print(df_deputados.isnull().mean().sort_values(ascending=False).to_string())

# Verifica a distribuição por partido (útil para validar depois)
if "siglaPartido" in df_deputados.columns:
    print("Deputados por partido (amostra 1 página):")
    print(df_deputados["siglaPartido"].value_counts().to_string())

# Verifica a distribuição por UF
if "siglaUf" in df_deputados.columns:
    print("Deputados por UF (amostra 1 página):")
    print(df_deputados["siglaUf"].value_counts().to_string())

Campos e % de nulos:
id               0.0
uri              0.0
nome             0.0
siglaPartido     0.0
uriPartido       0.0
siglaUf          0.0
idLegislatura    0.0
urlFoto          0.0
email            0.0
Deputados por partido (amostra 1 página):
siglaPartido
PL               18
PP               15
PT               14
MDB              12
REPUBLICANOS      8
PSD               6
UNIÃO             5
PSDB              4
PV                4
PDT               3
CIDADANIA         2
PODE              2
PSOL              2
NOVO              1
PCdoB             1
REDE              1
PSB               1
SOLIDARIEDADE     1
Deputados por UF (amostra 1 página):
siglaUf
SP    16
BA     9
RS     8
RJ     8
MG     6
PE     5
AM     4
PA     4
CE     4
PR     4
MA     4
SC     4
AP     3
TO     3
GO     2
PB     2
DF     2
AL     2
PI     2
RN     2
MS     2
RR     1
ES     1
AC     1
MT     1


In [13]:
# Detalhes de UM deputado — o endpoint /deputados/{id} tem muito mais campos
id_exemplo = df_deputados["id"].iloc[0]
raw_detalhe = get_endpoint(f"/deputados/{id_exemplo}")

print("\nCampos disponíveis no detalhe do deputado:")
print(json.dumps(raw_detalhe.get("dados", {}), indent=2, ensure_ascii=False))

[200] https://dadosabertos.camara.leg.br/api/v2/deputados/204379
  → 13 itens retornados nessa página

Campos disponíveis no detalhe do deputado:
{
  "id": 204379,
  "uri": "https://dadosabertos.camara.leg.br/api/v2/deputados/204379",
  "nomeCivil": "ACÁCIO DA SILVA FAVACHO NETO",
  "ultimoStatus": {
    "id": 204379,
    "uri": "https://dadosabertos.camara.leg.br/api/v2/deputados/204379",
    "nome": "Acácio Favacho",
    "siglaPartido": "MDB",
    "uriPartido": null,
    "siglaUf": "AP",
    "idLegislatura": 57,
    "urlFoto": "https://www.camara.leg.br/internet/deputado/bandep/204379.jpg",
    "email": null,
    "data": "2023-02-01",
    "nomeEleitoral": "Acácio Favacho",
    "gabinete": {
      "nome": "414",
      "predio": "4",
      "sala": "414",
      "andar": "4",
      "telefone": "3215-5414",
      "email": "dep.acaciofavacho@camara.leg.br"
    },
    "situacao": "Exercício",
    "condicaoEleitoral": "Titular",
    "descricaoStatus": null
  },
  "cpf": "74287028287",
  "sex

In [14]:
#Inspeciona os 'links' - a API usa HATEOAS para paginação
print("Links de paginação:")
for link in raw_deputados.get("links", []):
    print(f" {link}")

Links de paginação:
 {'rel': 'self', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=1&itens=100&ordem=ASC&ordenarPor=nome'}
 {'rel': 'next', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?ordem=ASC&ordenarPor=nome&pagina=2&itens=100'}
 {'rel': 'first', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?ordem=ASC&ordenarPor=nome&pagina=1&itens=100'}
 {'rel': 'last', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?ordem=ASC&ordenarPor=nome&pagina=6&itens=100'}


### 📝 Observações sobre `/deputados`

> - Campos no endpoint de lista: `id`, `uri`, `nome`, `siglaPartido`, `uriPartido`, `siglaUf`, `idLegislatura`, `urlFoto` `email`              `
> - Campos adicionais no endpoint de detalhe (`/deputados/{id}`): "id": 204379,
  `uri`,
  `nomeCivil`,
  `ultimoStatus`,
    `id`,
    `uri`,
    `nome`,
    `siglaPartido`,
    `uriPartido`,
    `siglaUf`,
    `idLegislatura`,
    `urlFoto`,
    `email`,
    `data`,
    `nomeEleitoral`,
    `gabinete`,
      `nome`,
      `predio`,
      `sala`,
      `andar`,
      `telefone`,
      `email`,
    `situacao`,
    `condicaoEleitoral`,
    `descricaoStatus`,
  `cpf`,
  `sexo`,
  `urlWebsite`,
  `redeSocial`,
  `dataNascimento`,
  `dataFalecimento`,
  `ufNascimento`,
  `municipioNascimento`,
  `escolaridade`
> - Total de deputados: `513`
> - Quantas páginas com `itens=100`: `6 páginas, sendo 5 com 100 itens e a sexta com 13`
> - Campos nulos recorrentes: `0`
> - Chave de relacionamento para outras tabelas: `id` (inteiro)
> - Decisão: vou usar lista ou detalhe? Lista é suficiente ou preciso de ambos? `Precisaremos de ambos. Algumas informações como a situação atual do deputa é importante e está nos detalhes`

---
## 4. Explorando `/proposicoes`

O endpoint mais importante para o produto final.  
Aqui entram: PLs, PDCs, MPVs, Requerimentos — tudo que tramita na Câmara.  
Este é o endpoint que vai receber a camada de IA (classificação + resumo).

In [16]:
# Proposições dos últimos 30 dias
# atenção: a API exige dataInicio e dataFim no formato YYYY-MM-DD
params_prop = {
    "dataInicio": DATA_INICIO,
    "dataFim":    DATA_FIM,
    "pagina":     1,
    "itens":      100,
    "ordem":      "DESC",
    "ordenarPor": "id"
}

raw_prop = get_endpoint("/proposicoes", params=params_prop)
df_prop  = normalizar(raw_prop)

print(f"Shape (1 página, 100 itens): {df_prop.shape}")
df_prop.head()

[200] https://dadosabertos.camara.leg.br/api/v2/proposicoes?dataInicio=2026-04-24&dataFim=2026-05-24&pagina=1&itens=100&ordem=DESC&ordenarPor=id
  → 100 itens retornados nessa página
Shape (1 página, 100 itens): (100, 8)


,id,uri,siglaTipo,codTipo,numero,ano,ementa,dataApresentacao
0,2627335,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2627335,REQ,310,3079,2026,"Requer a retirada do apoiamento à Emenda Nº 2, na PEC Nº 221 de 2019.",2026-05-22T18:57
1,2627334,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2627334,PL,139,2568,2026,"Altera o inciso VI do art. 20 da Lei nº 8.036, de 11 de maio de 1990, para r...",2026-05-22T18:16
2,2627333,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2627333,PL,139,2567,2026,"Confere ao Município de Mangaratiba, no Estado do Rio de Janeiro, o título d...",2026-05-22T17:50
3,2627332,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2627332,SBT,255,1,0,"Institui o Dia Nacional do Orgulho Rubro-Negro, a ser celebrado anualmente e...",2026-05-22T15:48
4,2627331,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2627331,PRL,190,1,0,"Parecer do Relator, Dep. Pastor Henrique Vieira (PSOL-RJ), pela aprovação de...",2026-05-22T15:48


In [17]:
# Analisa tipos e nulos
print("Tipos de dado:")
print(df_prop.dtypes.to_string())
print("\nCampos e % de nulos:")
print(df_prop.isnull().mean().sort_values(ascending=False).to_string())

Tipos de dado:
id                  int64
uri                   str
siglaTipo             str
codTipo             int64
numero              int64
ano                 int64
ementa                str
dataApresentacao      str

Campos e % de nulos:
id                  0.0
uri                 0.0
siglaTipo           0.0
codTipo             0.0
numero              0.0
ano                 0.0
ementa              0.0
dataApresentacao    0.0


In [18]:
# Olha o raw JSON de uma proposição — a ementa está no campo correto?
print("Exemplo de proposição (raw JSON):")
print(json.dumps(raw_prop["dados"][0], indent=2, ensure_ascii=False))

Exemplo de proposição (raw JSON):
{
  "id": 2627335,
  "uri": "https://dadosabertos.camara.leg.br/api/v2/proposicoes/2627335",
  "siglaTipo": "REQ",
  "codTipo": 310,
  "numero": 3079,
  "ano": 2026,
  "ementa": "Requer a retirada do  apoiamento à Emenda Nº 2, na PEC Nº 221 de 2019.",
  "dataApresentacao": "2026-05-22T18:57"
}


In [19]:
# Detalhe de UMA proposição — quais campos extras existem?
id_prop_exemplo = df_prop["id"].iloc[0]
raw_prop_detalhe = get_endpoint(f"/proposicoes/{id_prop_exemplo}")

print(f"\nDetalhe da proposição id={id_prop_exemplo}:")
print(json.dumps(raw_prop_detalhe.get("dados", {}), indent=2, ensure_ascii=False))

[200] https://dadosabertos.camara.leg.br/api/v2/proposicoes/2627335
  → 21 itens retornados nessa página

Detalhe da proposição id=2627335:
{
  "id": 2627335,
  "uri": "https://dadosabertos.camara.leg.br/api/v2/proposicoes/2627335",
  "siglaTipo": "REQ",
  "codTipo": 310,
  "numero": 3079,
  "ano": 2026,
  "ementa": "Requer a retirada do  apoiamento à Emenda Nº 2, na PEC Nº 221 de 2019.",
  "dataApresentacao": "2026-05-22T18:57",
  "uriOrgaoNumerador": null,
  "statusProposicao": {
    "dataHora": "2026-05-22T18:57",
    "sequencia": 1,
    "siglaOrgao": "PLEN",
    "uriOrgao": "https://dadosabertos.camara.leg.br/api/v2/orgaos/180",
    "uriUltimoRelator": null,
    "regime": ".",
    "descricaoTramitacao": "Apresentação de Proposição",
    "codTipoTramitacao": "100",
    "descricaoSituacao": null,
    "codSituacao": null,
    "despacho": "Apresentação do REQ n. 3079/2026 (Requerimento de Inclusão ou Retirada de Assinatura em Proposição de Iniciativa Coletiva Obrigatória), pelo Deputad

In [20]:
# Verifica: a ementa é longa o suficiente para embedding?
if "ementa" in df_prop.columns:
    print("Comprimento médio das ementas (caracteres):")
    print(df_prop["ementa"].str.len().describe())
    print("\n--- Exemplo de ementa longa ---")
    print(df_prop["ementa"].str.len().idxmax())
    idx_max = df_prop["ementa"].str.len().idxmax()
    print(df_prop.loc[idx_max, "ementa"])

Comprimento médio das ementas (caracteres):
count    100.0000
mean     114.4500
std      132.9475
min        0.0000
25%       43.7500
50%       68.0000
75%      128.5000
max      787.0000
Name: ementa, dtype: float64

--- Exemplo de ementa longa ---
43
Nos termos do art. 66 da Constituição, comunico que sancionei o Projeto de Lei n2 2.083, de 2022, que "Altera a Lei n 7.210, de 1]. de julho de 1984 (Lei de Execução Penal), para estabelecer medidas destinadas a reforçar a proteção da mulher vítima de violência doméstica e familiar, especialmente contra a reiteração de ameaça ou de violência perpetrada por agressores condenados ou submetidos a prisão provisória; e a Lei nQ 9Â55, de 7 de abril de 1997 (Lei dos Crimes de Tortura), para prever como modalidade de tortura a submissão reiterada da mulher a intenso sofrimento físico ou mental, no contexto de violência doméstica e familiar.". Para o arquivo do Congresso Nacional, restituo, nesta oportunidade, autógrafo do texto ora convertido na

In [21]:
# Tipos de proposição — que siglaTipo existe nos últimos 30 dias?
if "siglaTipo" in df_prop.columns:
    print("Tipos de proposição mais comuns:")
    print(df_prop["siglaTipo"].value_counts().to_string())

Tipos de proposição mais comuns:
siglaTipo
PRL      37
PAR      20
REQ       8
PL        7
SBR       6
SBT-A     6
EMC-A     4
SBT       3
MSC       3
SBE-A     3
RIC       1
EMR       1
PDL       1


In [22]:
# VOLUME — quantas proposições existem nos últimos 30 dias?
# Faz 3 páginas para estimar
totais = []
for p in range(1, 4):
    r = get_endpoint("/proposicoes", params={**params_prop, "pagina": p, "itens": 100})
    n = len(r.get("dados", []))
    totais.append(n)
    if n < 100:
        print(f"  → Última página encontrada na página {p}")
        break

print(f"\nItens por página: {totais}")
print(f"Total estimado (3 páginas): {sum(totais)}")
print("Conclusão: se todas as páginas têm 100 itens, há mais dados — o loop de paginação é obrigatório.")

[200] https://dadosabertos.camara.leg.br/api/v2/proposicoes?dataInicio=2026-04-24&dataFim=2026-05-24&pagina=1&itens=100&ordem=DESC&ordenarPor=id
  → 100 itens retornados nessa página
[200] https://dadosabertos.camara.leg.br/api/v2/proposicoes?dataInicio=2026-04-24&dataFim=2026-05-24&pagina=2&itens=100&ordem=DESC&ordenarPor=id
  → 100 itens retornados nessa página
[200] https://dadosabertos.camara.leg.br/api/v2/proposicoes?dataInicio=2026-04-24&dataFim=2026-05-24&pagina=3&itens=100&ordem=DESC&ordenarPor=id
  → 100 itens retornados nessa página

Itens por página: [100, 100, 100]
Total estimado (3 páginas): 300
Conclusão: se todas as páginas têm 100 itens, há mais dados — o loop de paginação é obrigatório.


### 📝 Observações sobre `/proposicoes`

> - Volume aproximado por 30 dias: `+300`
> - Campo da ementa (para embedding): `ementa` — comprimento médio: `114`
> - Campos disponíveis no detalhe que não aparecem na lista: `"statusProposicao": {
    "dataHora": "2026-05-22T18:57",
    "sequencia": 1,
    "siglaOrgao": "PLEN",
    "uriOrgao": "https://dadosabertos.camara.leg.br/api/v2/orgaos/180",
    "uriUltimoRelator": null,
    "regime": ".",
    "descricaoTramitacao": "Apresentação de Proposição",
    "codTipoTramitacao": "100",
    "descricaoSituacao": null,
    "codSituacao": null,
...
  "urnFinal": null,
  "texto": null,
  "justificativa": null
}`
> - Tipos de proposição mais comuns: `PRL`
> - Campos nulos recorrentes: `NA`
> - Chave de relacionamento com votações: `id`
> - Decisão: vou usar só a ementa para o embedding, ou também `ementaDetalhada`? `ementaDetalhada`
> - Decisão: vou filtrar por `siglaTipo` (só PLs) ou pegar tudo? `Pegar tudo e depois filtrar PL e PEC`